In [1]:
# 01 문제정의
# 02 데이터가져오기
# 03 EDA
# 04 데이터전처리
# 05 검증데이터 분할 및 학습
# 06 테스트 및 검증지표 확인
# 07 Model 내보내기 , 테스트파일 내보내기

In [2]:
## 고객별 신용등급 분류
## 타깃(라벨, y값) : Credit_Score  ->  0- Poor , 1 - Standard 2 - Good (3개 등급)
##
## 목적 : 신규 고객 정보가 등록되었을 때 고객의 신용등급을 자동으로 분류해 저장한다.
##
## 정할 것 4가지
##   1) 무엇을 맞히나  : Credit_Score (신용등급)
##   2) 답이 몇 종류인가 : 3종류  ->  다중분류  (2종이면 이진분류, 연속 숫자면 회귀)
##   3) 양성(1)은?     : 해당 없음. 세 등급이 대등하므로 '양성'이라는 개념이 없다
##   4) 어떻게 채점하나 : 정확도 + f1(macro)
##                      ROC-AUC는 못 쓴다 - '양성 vs 음성' 구도가 없기 때문
##
## 3)이 없어지니 4)가 바뀐다는 점이 04 이진분류와의 핵심 차이.

In [3]:
# ----------------------------
# 02 데이터가져오기
# ----------------------------

In [4]:
import pandas as pd
train = pd.read_csv('train.csv', low_memory=False) # 학습용 데이터 읽기 (low_memory=False: 타입 혼합 경고 방지)
test  = pd.read_csv('test.csv', low_memory=False)  # 학습용 데이터 읽기 (low_memory=False: 타입 혼합 경고 방지)
print('train:', train.shape, '| test:', test.shape)

train: (100000, 28) | test: (50000, 27)


In [5]:
# ----------------------------
# 03 EDA
# ----------------------------

In [6]:
train.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [7]:
train['Credit_Score'].unique()

array(['Good', 'Standard', 'Poor'], dtype=object)

In [8]:
test.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance
0,0x160a,CUS_0xd40,September,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,2022.0,Good,809.98,35.030402,22 Years and 9 Months,No,49.574949,236.64268203272135,Low_spent_Small_value_payments,186.26670208571772
1,0x160b,CUS_0xd40,October,Aaron Maashoh,24,821-00-0265,Scientist,19114.12,1824.843333,3,...,4.0,Good,809.98,33.053114,22 Years and 10 Months,No,49.574949,21.465380264657146,High_spent_Medium_value_payments,361.44400385378196
2,0x160c,CUS_0xd40,November,Aaron Maashoh,24,821-00-0265,Scientist,19114.12,1824.843333,3,...,4.0,Good,809.98,33.811894,NaN,No,49.574949,148.23393788500925,Low_spent_Medium_value_payments,264.67544623342997
3,0x160d,CUS_0xd40,December,Aaron Maashoh,24_,821-00-0265,Scientist,19114.12,NaN,3,...,4.0,Good,809.98,32.430559,23 Years and 0 Months,No,49.574949,39.08251089460281,High_spent_Medium_value_payments,343.82687322383634
4,0x1616,CUS_0x21b1,September,Rick Rothackerj,28,004-07-5839,_______,34847.84,3037.986667,2,...,5.0,Good,605.03,25.926822,27 Years and 3 Months,No,18.816215,39.684018417945296,High_spent_Large_value_payments,485.2984336755923


In [9]:
train.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


In [10]:
test.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,42502.000000,50000.000000,50000.000000,50000.000000,50000.000000,48965.000000,50000.000000,50000.000000
mean,4182.004291,16.838260,22.921480,68.772640,21.052640,30.080200,32.279581,1491.304305
std,3174.109304,116.396848,129.314804,451.602363,14.860397,196.984121,5.106238,8595.647887
min,303.645417,-1.000000,0.000000,1.000000,-5.000000,0.000000,20.509652,0.000000
25%,1625.188333,3.000000,4.000000,8.000000,10.000000,4.000000,28.061040,32.222388
50%,3086.305000,6.000000,5.000000,13.000000,18.000000,7.000000,32.280390,74.733349
75%,5934.189094,7.000000,7.000000,20.000000,28.000000,10.000000,36.468591,176.157491
max,15204.633333,1798.000000,1499.000000,5799.000000,67.000000,2593.000000,48.540663,82398.000000


In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [12]:
test.info();

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        50000 non-null  object 
 1   Customer_ID               50000 non-null  object 
 2   Month                     50000 non-null  object 
 3   Name                      44985 non-null  object 
 4   Age                       50000 non-null  object 
 5   SSN                       50000 non-null  object 
 6   Occupation                50000 non-null  object 
 7   Annual_Income             50000 non-null  object 
 8   Monthly_Inhand_Salary     42502 non-null  float64
 9   Num_Bank_Accounts         50000 non-null  int64  
 10  Num_Credit_Card           50000 non-null  int64  
 11  Interest_Rate             50000 non-null  int64  
 12  Num_of_Loan               50000 non-null  object 
 13  Type_of_Loan              44296 non-null  object 
 14  Delay_

In [13]:
train.isnull().sum()

ID                              0
Customer_ID                     0
Month                           0
Name                         9985
Age                             0
SSN                             0
Occupation                      0
Annual_Income                   0
Monthly_Inhand_Salary       15002
Num_Bank_Accounts               0
Num_Credit_Card                 0
Interest_Rate                   0
Num_of_Loan                     0
Type_of_Loan                11408
Delay_from_due_date             0
Num_of_Delayed_Payment       7002
Changed_Credit_Limit            0
Num_Credit_Inquiries         1965
Credit_Mix                      0
Outstanding_Debt                0
Credit_Utilization_Ratio        0
Credit_History_Age           9030
Payment_of_Min_Amount           0
Total_EMI_per_month             0
Amount_invested_monthly      4479
Payment_Behaviour               0
Monthly_Balance              1200
Credit_Score                    0
dtype: int64

In [14]:
test.isnull().sum()

ID                             0
Customer_ID                    0
Month                          0
Name                        5015
Age                            0
SSN                            0
Occupation                     0
Annual_Income                  0
Monthly_Inhand_Salary       7498
Num_Bank_Accounts              0
Num_Credit_Card                0
Interest_Rate                  0
Num_of_Loan                    0
Type_of_Loan                5704
Delay_from_due_date            0
Num_of_Delayed_Payment      3498
Changed_Credit_Limit           0
Num_Credit_Inquiries        1035
Credit_Mix                     0
Outstanding_Debt               0
Credit_Utilization_Ratio       0
Credit_History_Age          4470
Payment_of_Min_Amount          0
Total_EMI_per_month            0
Amount_invested_monthly     2271
Payment_Behaviour              0
Monthly_Balance              562
dtype: int64

In [15]:
# 타깃이 몇 종류인가  ->  이 한 줄이 '문제 유형'을 확정한다
train['Credit_Score'].unique()      # [1 2 3]  ->  3종류이므로 다중분류

array(['Good', 'Standard', 'Poor'], dtype=object)

In [16]:
# 등급별 개수 - 쏠림(불균형)이 있는지 확인
train['Credit_Score'].value_counts()   # 2:5237  1:2978  3:1785  (약 3배 차이)

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

In [17]:
# ----------------------------
# 04 데이터전처리
# ----------------------------

In [18]:
# 데이터 오염여부 확인 + 자동 수정

In [19]:
# [1] 오염 체크
print('===== 오염 체크 =====')                                                    # 제목 출력
for c in train.select_dtypes(include='object').columns:                          # 글자(object) 컬럼만 하나씩 반복
    ratio = pd.to_numeric(train[c].astype(str).str.replace('_',''), errors='coerce').notna().mean()   # 밑줄(_) 지우고 숫자변환 시도 → 성공한 값의 비율 계산
    print(f'{c:26s} 숫자변환가능 {ratio:5.0%}', '  <== 숫자 오염!' if ratio>=0.9 else '')   # 컬럼명과 비율 출력, 90% 이상이면 오염 표시

# [2] 자동 수정 (90% 이상 숫자면 변환)
for c in train.select_dtypes(include='object').columns:                          # 글자 컬럼만 다시 하나씩 반복
    if pd.to_numeric(train[c].astype(str).str.replace('_',''), errors='coerce').notna().mean() >= 0.9:   # 숫자변환 성공 비율이 90% 이상이면
        train[c] = pd.to_numeric(train[c].astype(str).str.replace('_',''), errors='coerce')   # train 컬럼을 숫자로 변환 (실패값은 NaN)
        test[c]  = pd.to_numeric(test[c].astype(str).str.replace('_',''), errors='coerce')     # test도 똑같이 숫자로 변환

===== 오염 체크 =====
ID                         숫자변환가능    0% 
Customer_ID                숫자변환가능    0% 
Month                      숫자변환가능    0% 
Name                       숫자변환가능    0% 
Age                        숫자변환가능  100%   <== 숫자 오염!
SSN                        숫자변환가능    0% 
Occupation                 숫자변환가능    0% 
Annual_Income              숫자변환가능  100%   <== 숫자 오염!
Num_of_Loan                숫자변환가능  100%   <== 숫자 오염!
Type_of_Loan               숫자변환가능    0% 
Num_of_Delayed_Payment     숫자변환가능   93%   <== 숫자 오염!
Changed_Credit_Limit       숫자변환가능   98%   <== 숫자 오염!
Credit_Mix                 숫자변환가능    0% 
Outstanding_Debt           숫자변환가능  100%   <== 숫자 오염!
Credit_History_Age         숫자변환가능    0% 
Payment_of_Min_Amount      숫자변환가능    0% 
Amount_invested_monthly    숫자변환가능   96%   <== 숫자 오염!
Payment_Behaviour          숫자변환가능    0% 
Monthly_Balance            숫자변환가능   99%   <== 숫자 오염!
Credit_Score               숫자변환가능    0% 


In [20]:
# 오염도 검증 LIB 보고서 사용 - (선택)
# pip install ydata-profiling
from ydata_profiling import ProfileReport
ProfileReport(train, title="Credit Report").to_file("report.html")

/tmp/ipykernel_1687/3628142581.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 28/28 [00:02<00:00, 13.87it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
# 결측치 처리

In [23]:
# 결측치 확인 (train / test 둘 다)
print('train 결측:', train.isnull().sum().sum())
print('test  결측:', test.isnull().sum().sum())
# 이 데이터는 둘 다 0 - 채울 게 없다.
# '전처리를 안 한 것'이 아니라 '확인해 보니 할 게 없었던 것'이다.

train 결측: 62162
test  결측: 31112


In [24]:
# cat_cols = [c for c in train.columns if train[c].dtype == 'object']
# print('범주형:', cat_cols)

null_series = train.isnull().sum()

null_object_series = null_series[(null_series>0) & (train.dtypes=='object')].index
null_object_series

Index(['Name', 'Type_of_Loan', 'Credit_History_Age'], dtype='object')

In [25]:
# cat_cols_float = [c for c in train.columns if train[c].dtype == 'float64']
# print('수치형:', cat_cols_float)

null_float64_series = null_series[(null_series>0) & (train.dtypes=='float64')].index
null_float64_series


Index(['Monthly_Inhand_Salary', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries',
       'Amount_invested_monthly', 'Monthly_Balance'],
      dtype='object')

In [26]:
num_cols = null_float64_series  # 수치형 데이터
cat_cols = null_object_series # 범주형 데이터

# 결측치 채우기 - 이 데이터는 결측이 0이라 실제로 채워지는 값이 없다.
# (결측이 있는 데이터라면 아래처럼 train 기준값으로 train/test를 함께 채운다)

for c in num_cols:
    m = train[c].median()            # train 기준 중앙값
    train[c] = train[c].fillna(m)
    test[c] = test[c].fillna(m)
    
for c in cat_cols:
    m = train[c].mode()[0]           # train 기준 최빈값
    train[c] = train[c].fillna(m)
    test[c] = test[c].fillna(m)
    
print('남은 결측치:', train.isnull().sum().sum(), test.isnull().sum().sum())

#[CF] 
# 결측이 있는 열을 타입별로 뽑아 두는 방법 (이 데이터는 결측이 없어 빈 결과가 나온다)
# null_series = train.isnull().sum()

# num_like = train.select_dtypes(include='number').columns      # 수치형
# cat_like = train.select_dtypes(exclude='number').columns      # 글자형

# null_num_cols = [c for c in num_like if null_series[c] > 0]
# null_cat_cols = [c for c in cat_like if null_series[c] > 0]
# print('결측 수치형:', null_num_cols)   # []
# print('결측 글자형:', null_cat_cols)   # []


남은 결측치: 0 0


In [27]:
train.info()
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      100000 non-null  object 
 4   Age                       100000 non-null  int64  
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     100000 non-null  float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  int64  
 13  Type_of_Loan              100000 non-null  ob

In [28]:
# y값 분리

In [29]:
## 0- POOR , 1 - STANDARD 2 - GOOD
y = train['Credit_Score'].map({'Poor': 0, 'Standard': 1, 'Good':2})   # 문자 → 0/1
y

# [참고] 정답이 'Poor'/'Standard'/'Good' 같은 글자였다면 두 가지 선택이 있다.
#   1) 그대로 둔다      - sklearn은 글자 정답을 그대로 받는다
#   2) map으로 숫자화   - y = train['Credit_Score'].map({'Poor':0,'Standard':1,'Good':2})
# 어느 쪽이든 되지만, 등급처럼 순서가 있으면 숫자로 두는 편이 해석에 편하다.

0        2
1        2
2        2
3        2
4        2
        ..
99995    0
99996    0
99997    0
99998    1
99999    0
Name: Credit_Score, Length: 100000, dtype: int64

In [30]:
train = train.drop(columns=['ID', 'Customer_ID','Credit_Score'])
train.shape

(100000, 25)

In [31]:
test = test.drop(columns=['ID', 'Customer_ID'])
test.shape

(50000, 25)

In [32]:

# 원핫 인코딩 불가(너무 많은클래스) -> 라벨 인코딩 
# ---------------------
# 기존 레이블인코딩
# ---------------------
from sklearn.preprocessing import LabelEncoder

cols = train.columns[train.dtypes == object]

for col in cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col],test[col]] ,axis=0)) # 행합치기
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

print(train[cols].head())

   Month  Name    SSN  Occupation  Type_of_Loan  Credit_Mix  \
0      4    84  10205          12           128           3   
1      3    84  10205          12           128           1   
2      7    84  10205          12           128           1   
3      0    84  10205          12           128           1   
4      8    84  10205          12           128           1   

   Credit_History_Age  Payment_of_Min_Amount  Payment_Behaviour  
0                 180                      1                  3  
1                  86                      1                  4  
2                 184                      1                  5  
3                 185                      1                  6  
4                 186                      1                  2  


In [33]:
train

,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,...,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance
0,4,84,23,10205,12,19114.12,1824.843333,3,4,3,...,4.0,3,809.98,26.822620,180,1,49.574949,80.415295,3,312.494089
1,3,84,23,10205,12,19114.12,3093.745000,3,4,3,...,4.0,1,809.98,31.944960,86,1,49.574949,118.280222,4,284.629162
2,7,84,-500,10205,12,19114.12,3093.745000,3,4,3,...,4.0,1,809.98,28.609352,184,1,49.574949,81.699521,5,331.209863
3,0,84,23,10205,12,19114.12,3093.745000,3,4,3,...,4.0,1,809.98,31.377862,185,1,49.574949,199.458074,6,223.451310
4,8,84,23,10205,12,19114.12,1824.843333,3,4,3,...,4.0,1,809.98,24.797347,186,1,49.574949,41.420153,2,341.489231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,6528,25,1009,9,39628.99,3359.415833,4,6,7,...,3.0,3,502.38,34.663572,307,1,35.104023,60.971333,1,479.866228
99996,8,6528,25,1009,9,39628.99,3359.415833,4,6,7,...,3.0,3,502.38,40.565631,308,1,35.104023,54.185950,2,496.651610
99997,6,6528,25,1009,9,39628.99,3359.415833,4,6,5729,...,3.0,1,502.38,41.255522,309,1,35.104023,24.028477,1,516.809083
99998,5,6528,25,1009,9,39628.99,3359.415833,4,6,7,...,3.0,1,502.38,33.638208,310,1,35.104023,251.672582,4,319.164979


In [34]:
# ----------------------------
# 05 검증데이터 분할 및 학습
# ----------------------------

In [35]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train, y, test_size=0.2, random_state=0, stratify=y)
print(X_tr.shape, X_val.shape)   # (8000, 29) (2000, 29)

(80000, 25) (20000, 25)


In [36]:
# ----------------------------
# 06 테스트 및 검증지표 확인
# ----------------------------

In [37]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_tr, y_tr)
print('클래스:', rf.classes_)   # [1 2 3]

클래스: [0 1 2]


In [38]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

pred = rf.predict(X_val)
print('정확도    :', round(accuracy_score(y_val, pred), 4))                  # 0.7285
print('f1(macro) :', round(f1_score(y_val, pred, average='macro'), 4))       # 0.7107
print()
print(classification_report(y_val, pred))
#
# average='macro'는 다중분류에서 필수 - 안 주면 에러가 난다.
#   ("클래스별 f1 3개를 어떻게 합칠지 네가 정하라"는 뜻)

정확도    : 0.7885
f1(macro) : 0.7751

              precision    recall  f1-score   support

           0       0.78      0.81      0.79      5799
           1       0.81      0.80      0.81     10635
           2       0.74      0.71      0.72      3566

    accuracy                           0.79     20000
   macro avg       0.78      0.77      0.78     20000
weighted avg       0.79      0.79      0.79     20000



In [43]:
# macro vs weighted 비교 - 이 단원에서 가장 중요한 대목
print('f1(macro)   :', round(f1_score(y_val, pred, average='macro'), 4))     # 0.7107
print('f1(weighted):', round(f1_score(y_val, pred, average='weighted'), 4))  # 0.7295
#
# weighted가 더 높다. 개수로 가중평균해서 제일 많은 2등급(f1 0.76)이 점수를 끌어올리고,
# 제일 적고 어려운 3등급(f1 0.64)은 묻히기 때문이다.
# macro는 개수를 무시하고 클래스마다 1표씩 준다.
# 두 값의 차이(약 0.02)가 곧 '소수 클래스가 약하다'는 신호다.

f1(macro)   : 0.7751
f1(weighted): 0.7883


In [44]:
# ----------------------------
# 07 Model 내보내기 , 테스트파일 내보내기
# ----------------------------

In [45]:
import joblib

joblib.dump(rf, 'model.pkl')          # 학습된 모델을 파일로
print('모델 저장 완료: model.pkl')

loaded = joblib.load('model.pkl')     # 다시 불러오기
print('불러온 모델 예측 일치:', (loaded.predict(X_val) == pred).all())

모델 저장 완료: model.pkl
불러온 모델 예측 일치: True


In [ ]:
# ----------------------------
# 08 기타
# ----------------------------

In [42]:
# 이상치 확인
train.select_dtypes('number').plot(kind='box', subplots=True, figsize=(15,8))
plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

# 박스플롯(Box Plot) 해석 가이드
#### 박스플롯은 데이터의 **분포**와 **이상치**를 한눈에 보여주는 그림입니다.
```
    │  ← 최댓값 (위쪽 수염 끝)
    │
┌───┴───┐  ← Q3 (75% 지점, 박스 위쪽 경계)
│       │
│───────│  ← 중앙값 (Median, 50% 지점)
│       │
└───┬───┘  ← Q1 (25% 지점, 박스 아래쪽 경계)
    │
    │  ← 최솟값 (아래쪽 수염 끝)
    ●  ← 이상치 (수염 밖의 점)
```
## 각 부분의 의미

| 요소 | 의미 |
|------|------|
| **박스(상자)** | 데이터의 가운데 50%가 모여있는 구간 (Q1 ~ Q3) |
| **박스 안 선** | 중앙값(Median). 평균이 아니라 딱 가운데 값 |
| **박스 높이(IQR)** | Q3 - Q1. 데이터가 얼마나 퍼져있는지 나타냄 |
| **수염(whisker)** | 정상 범위의 최소/최대. 보통 Q1-1.5×IQR ~ Q3+1.5×IQR |
| **점(flier)** | 수염을 벗어난 **이상치(outlier)** |

## 이렇게 읽으면 됩니다

1. **중앙값 선의 위치** → 데이터의 중심이 어디인지 확인
2. **박스가 크다** → 데이터가 넓게 퍼져있음 (변동 큼)
   **박스가 작다** → 데이터가 촘촘히 모여있음 (변동 작음)
3. **중앙값 선이 박스 가운데가 아니라 한쪽에 치우침** → 분포가 비대칭(왜곡됨)
   - 선이 아래쪽에 있으면 → 위로 긴 꼬리 (오른쪽으로 치우침)
   - 선이 위쪽에 있으면 → 아래로 긴 꼬리 (왼쪽으로 치우침)
4. **박스 밖의 점들** → 이상치. 많으면 처리 방법을 고민해야 함

## 머신러닝 관점에서 볼 점

- **이상치(점)가 많은 컬럼** → 제거/변환(로그 등)을 고려
- **한쪽으로 심하게 치우친 컬럼** → 로그 변환으로 분포를 펴주면 도움됨
- **선형/로지스틱 회귀 사용 시** → 이상치에 민감하므로 주의
- **트리 기반 모델(랜덤포레스트, XGBoost)** → 이상치에 강건해서 덜 민감